In [ ]:
import os
import numpy as np
import librosa
from glob import glob
from tqdm import tqdm
import zipfile
import shutil

# Define paths
base_path = "/kaggle/input/fakeav/AUDIO/AUDIO"  # Adjust this path according to your input
output_base = "/kaggle/working/audio_mfcc_4class"
zip_output_path = "/kaggle/working/audio_mfcc_dataset.zip"

# Define 4 classes
AUDIO_CLASSES = {
    'FakeVideo-FakeAudio': 'fake_video_fake_audio',
    'FakeVideo-RealAudio': 'fake_video_real_audio', 
    'RealVideo-FakeAudio': 'real_video_fake_audio',
    'RealVideo-RealAudio': 'real_video_real_audio'
}

def organize_audio_files():
    """
    Organize audio files into 4 categories based on directory structure
    """
    print("=" * 60)
    print("STEP 1: ORGANIZING AUDIO FILES INTO 4 CLASSES")
    print("=" * 60)
    
    # Create output directories for all 4 classes
    class_files = {}
    
    for original_dir, class_name in AUDIO_CLASSES.items():
        class_dir = os.path.join(output_base, class_name)
        os.makedirs(class_dir, exist_ok=True)
        
        # Find all audio files for this class
        class_pattern = os.path.join(base_path, original_dir, "**", "*.wav")
        files = glob(class_pattern, recursive=True)
        
        # Also try without subdirectories
        if len(files) == 0:
            class_pattern = os.path.join(base_path, original_dir, "*.wav")
            files = glob(class_pattern, recursive=True)
        
        class_files[class_name] = files
        print(f"📁 {class_name}: Found {len(files)} audio files")
    
    # Check if we found any files
    total_files = sum(len(files) for files in class_files.values())
    if total_files == 0:
        print("⚠️  No audio files found. Checking available directories...")
        available_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
        print(f"Available directories: {available_dirs}")
        
        # Try to map available directories to our classes
        for available_dir in available_dirs:
            for original_dir, class_name in AUDIO_CLASSES.items():
                if original_dir.lower() in available_dir.lower() or available_dir.lower() in original_dir.lower():
                    class_pattern = os.path.join(base_path, available_dir, "**", "*.wav")
                    files = glob(class_pattern, recursive=True)
                    if len(files) == 0:
                        class_pattern = os.path.join(base_path, available_dir, "*.wav")
                        files = glob(class_pattern, recursive=True)
                    
                    if len(files) > 0:
                        class_files[class_name] = files
                        print(f"📁 {class_name} (mapped from {available_dir}): Found {len(files)} audio files")
    
    return class_files

def extract_mfcc_features(audio_files, output_dir, class_name, max_files=None):
    """
    Extract MFCC features from audio files and save as .npy
    """
    print(f"🎵 Extracting MFCC features for {class_name}...")
    
    if len(audio_files) == 0:
        print(f"⚠️  No files found for {class_name}")
        return 0
    
    if max_files:
        audio_files = audio_files[:max_files]
    
    saved_count = 0
    error_count = 0
    
    for i, audio_path in enumerate(tqdm(audio_files, desc=f"Processing {class_name}")):
        try:
            # Create output filename
            filename = os.path.basename(audio_path).replace('.wav', '')
            output_file = os.path.join(output_dir, f"{filename}_{class_name}.npy")
            
            # Skip if already exists
            if os.path.exists(output_file):
                saved_count += 1
                continue
            
            # Load audio and extract MFCC
            y, sr = librosa.load(audio_path, sr=16000, duration=10)  # Limit to 10 seconds
            
            # Skip if audio is too short
            if len(y) < sr * 0.5:  # Skip if less than 0.5 seconds
                continue
            
            # Extract MFCC features
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40, n_fft=2048, hop_length=512)
            
            # Ensure consistent shape by padding or truncating
            target_frames = 313  # Approximately 10 seconds at 16kHz with hop_length=512
            if mfcc.shape[1] < target_frames:
                # Pad with zeros
                pad_width = target_frames - mfcc.shape[1]
                mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')
            else:
                # Truncate
                mfcc = mfcc[:, :target_frames]
            
            # Save MFCC features
            np.save(output_file, mfcc)
            saved_count += 1
            
        except Exception as e:
            error_count += 1
            if error_count <= 5:  # Show first 5 errors
                print(f"Error processing {audio_path}: {str(e)}")
    
    print(f"✅ {class_name} processing completed.")
    print(f"   Saved: {saved_count}, Errors: {error_count}")
    
    return saved_count

def create_dataset_info():
    """Create a dataset info file"""
    info_content = """# Audio MFCC Dataset - 4 Classes

## Dataset Structure:
- fake_video_fake_audio/: MFCC features from fake video + fake audio
- fake_video_real_audio/: MFCC features from fake video + real audio  
- real_video_fake_audio/: MFCC features from real video + fake audio
- real_video_real_audio/: MFCC features from real video + real audio

## Feature Details:
- MFCC coefficients: 40
- Sampling rate: 16kHz
- Duration: 10 seconds (padded/truncated)
- Frame size: 313 frames
- Feature shape: (40, 313)

## File Format:
- Each .npy file contains MFCC features for one audio sample
- Filename format: {original_name}_{class_name}.npy

## Usage:
```python
import numpy as np
mfcc_features = np.load('filename.npy')
print(f"Shape: {mfcc_features.shape}")  # Should be (40, 313)
```
"""
    
    info_file = os.path.join(output_base, "README.md")
    with open(info_file, 'w') as f:
        f.write(info_content)
    
    print("📄 Created dataset README.md")

def create_zip_file():
    """Create a zip file containing all processed MFCC features"""
    print("=" * 60)
    print("STEP 3: CREATING ZIP FILE FOR DOWNLOAD")
    print("=" * 60)
    
    if os.path.exists(zip_output_path):
        os.remove(zip_output_path)
    
    with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add all files from output directory
        for root, dirs, files in os.walk(output_base):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, output_base)
                zipf.write(file_path, arcname)
                
    zip_size = os.path.getsize(zip_output_path) / (1024 * 1024)  # Size in MB
    print(f"📦 Zip file created: {zip_output_path}")
    print(f"📊 Zip file size: {zip_size:.2f} MB")
    
    return zip_output_path

def check_preprocessing_done():
    """Check if preprocessing is already completed"""
    if not os.path.exists(output_base):
        return False
    
    for class_name in AUDIO_CLASSES.values():
        class_dir = os.path.join(output_base, class_name)
        if not os.path.exists(class_dir):
            return False
        
        npy_files = glob(f"{class_dir}/*.npy")
        if len(npy_files) == 0:
            return False
    
    return True

def main():
    """Main processing function"""
    print("🎵 AUDIO MFCC EXTRACTION - 4 CLASS DATASET")
    print("=" * 60)
    
    # Check if preprocessing is done
    preprocessing_done = check_preprocessing_done()
    print(f"Preprocessing status: {'✅ COMPLETED' if preprocessing_done else '❌ NEEDED'}")
    
    if not preprocessing_done:
        # Step 1: Organize files
        class_files = organize_audio_files()
        
        # Step 2: Extract MFCC features
        print("\n" + "=" * 60)
        print("STEP 2: EXTRACTING MFCC FEATURES")
        print("=" * 60)
        
        total_processed = 0
        class_counts = {}
        
        for class_name, files in class_files.items():
            if len(files) > 0:
                count = extract_mfcc_features(
                    files, 
                    os.path.join(output_base, class_name), 
                    class_name, 
                    max_files=None  # Process all files
                )
                class_counts[class_name] = count
                total_processed += count
        
        # Create dataset info
        create_dataset_info()
        
        print(f"\n🎉 MFCC extraction completed!")
        print("📊 Summary:")
        for class_name, count in class_counts.items():
            print(f"   {class_name}: {count} samples")
        print(f"   Total samples: {total_processed}")
        
    else:
        print("=" * 60)
        print("STEP 2: MFCC EXTRACTION (SKIPPED - ALREADY EXISTS)")
        print("=" * 60)
        
        # Count existing files
        class_counts = {}
        total_files = 0
        for class_name in AUDIO_CLASSES.values():
            class_dir = os.path.join(output_base, class_name)
            if os.path.exists(class_dir):
                npy_files = glob(f"{class_dir}/*.npy")
                class_counts[class_name] = len(npy_files)
                total_files += len(npy_files)
        
        print("📊 Existing files:")
        for class_name, count in class_counts.items():
            print(f"   {class_name}: {count} samples")
        print(f"   Total samples: {total_files}")
    
    # Step 3: Create zip file
    zip_path = create_zip_file()
    
    print("\n" + "=" * 60)
    print("✅ PROCESSING COMPLETED!")
    print("=" * 60)
    print(f"📦 Download your dataset: {zip_path}")
    print(f"📁 Dataset structure:")
    for class_name in AUDIO_CLASSES.values():
        class_dir = os.path.join(output_base, class_name)
        if os.path.exists(class_dir):
            count = len(glob(f"{class_dir}/*.npy"))
            print(f"   └── {class_name}/: {count} .npy files")

if __name__ == "__main__":
    main()

🎵 AUDIO MFCC EXTRACTION - 4 CLASS DATASET
Preprocessing status: ❌ NEEDED
STEP 1: ORGANIZING AUDIO FILES INTO 4 CLASSES
📁 fake_video_fake_audio: Found 10835 audio files
📁 fake_video_real_audio: Found 9709 audio files
📁 real_video_fake_audio: Found 500 audio files
📁 real_video_real_audio: Found 500 audio files

STEP 2: EXTRACTING MFCC FEATURES
🎵 Extracting MFCC features for fake_video_fake_audio...


Processing fake_video_fake_audio: 100%|██████████| 10835/10835 [05:18<00:00, 34.01it/s]


✅ fake_video_fake_audio processing completed.
   Saved: 10835, Errors: 0
🎵 Extracting MFCC features for fake_video_real_audio...


Processing fake_video_real_audio:  77%|███████▋  | 7450/9709 [04:44<01:20, 28.02it/s]